In [ ]:
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.axes import Axes
from scipy import stats
from scipy.stats import ttest_rel

import utils

plt.style.use('default')

In [ ]:
METRICS = ['LINE_COVERAGE', 'BRANCH_COVERAGE', 'RAW_MUT_SCORE', 'COV_MUT_SCORE']
CI = 95  # 95 % CI
ALPHA = 1 - CI / 100.0

df = utils.union_read_csv('data/Gson_iterations.csv', 'data/Lang_iterations.csv', 'data/JacksonCore_iterations.csv')
df = df[df['RUN_COMPLETED'] == True].dropna(subset=['FEEDBACK_TYPE'])
df['RAW_MUT_SCORE'] = df['KILLED_MUTANTS'] * 1.0 / df['MUTANTS']
df['COV_MUT_SCORE'] = df['KILLED_MUTANTS'] * 1.0 / df['COVERED_MUTANTS']

In [ ]:
pivot = df.pivot(columns=['ITERATION'], index=['RUN_UUID', 'CLASS_NAME', 'FEEDBACK_TYPE'], values=METRICS)
for m in METRICS:
    for i in range(1, 11):
        pivot[(f'{m}_delta_prev', i)] = pivot[(m, i)] - pivot[(m, i-1)]
        pivot[(f'{m}_delta_base', i)] = pivot[(m, i)] - pivot[(m, 0)]

group_by = ['FEEDBACK_TYPE', 'ITERATION']
rows = []
for index, grouped in pivot.stack().groupby(group_by):
    row = {k: v for k, v in zip(group_by, index)}
    for m in METRICS:
        for baseline in ['prev', 'base']:
            mean, std, n, p_value = utils.diff_mean_std_n_p(grouped[f'{m}_delta_{baseline}'])
            row[f'{m}_delta_{baseline}_mean'] = mean
            row[f'{m}_delta_{baseline}_std'] = std
            row[f'{m}_delta_{baseline}_n'] = n
            row[f'{m}_delta_{baseline}_p'] = p_value
            row[f'{m}_delta_{baseline}_sig'] = p_value < ALPHA
    rows.append(row)

out = pd.DataFrame(rows).sort_values(group_by)
out = out[out['ITERATION'] > 0]

for ty in ['COVERAGE', 'MUTATION']:
    for baseline in ['prev', 'base']:
        curr = out[out['FEEDBACK_TYPE'] == ty][['ITERATION', *[f'{m}_delta_{baseline}_{z}' for m in METRICS for z in ['mean', 'std', 'n', 'p']]]]
        for m in METRICS:
            curr[f'{m}_delta_{baseline}_mean'] = curr[f'{m}_delta_{baseline}_mean'].apply(lambda x: f'{x:.3f}')
            curr[f'{m}_delta_{baseline}_std'] = curr[f'{m}_delta_{baseline}_std'].apply(lambda x: f'{x:.3f}')
            curr[f'{m}_delta_{baseline}_p'] = curr[f'{m}_delta_{baseline}_p'].apply(lambda x: '<0.001' if x < .001 else f'{x:.3f}')
        curr.to_csv(f'data/out/quality-diff-significance-{baseline}-{ty}.csv', index=False)

prev = out[['FEEDBACK_TYPE', 'ITERATION', *[f'{m}_delta_prev_{z}' for m in METRICS for z in ['mean', 'std', 'n', 'p', 'sig']]]]
base = out[['FEEDBACK_TYPE', 'ITERATION', *[f'{m}_delta_base_{z}' for m in METRICS for z in ['mean', 'std', 'n', 'p', 'sig']]]]
base

In [ ]:
import math

mean = .014856
std = .090611
n = 1236
se = std / math.sqrt(n)
t_stat = mean / se
p_two_sided = 2*(1-stats.t.cdf(abs(t_stat), df=n-1))
p_two_sided